# ***Build a Simple LLM Application with LCEL***

In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [1]:
# Open AI API Key and Open Source models--Llama3,Gemma2,mistral--Groq
import os
from dotenv import load_dotenv

In [3]:
# Load environment variables from a .env file
load_dotenv()

True

In [4]:
# Set up API keys
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ['HUGGINGFACEHUB_API_TOKEN'] = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [7]:
# Import necessary modules
from langchain_community.llms import HuggingFaceHub
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [13]:
# Load a lightweight, CPU-friendly LLM model from Hugging Face Hub
hf_model = HuggingFaceHub(repo_id="google/gemma-2b")  # Change model here for efficiency

In [14]:
# Load a Groq model (optional)
groq_model = ChatGroq(model="Gemma2-9b-It", groq_api_key=groq_api_key)

In [15]:
# Define a basic translation system
messages = [
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello, how are you?")
]

In [16]:
# Invoke Hugging Face model
result = hf_model.invoke(messages)
print("\n **Hugging Face Model Translation:**\n", result)

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)



 **Hugging Face Model Translation:**
 System: Translate the following from English to French
Human: Hello, how are you? How's it going?
Assistant: Bonjour! Je vais bien, merci. Et vous? Ca va?


In [17]:
# Parse output
parser = StrOutputParser()
parsed_result = parser.invoke(result)
print("\n**Parsed Translation Output:**\n", parsed_result)


**Parsed Translation Output:**
 System: Translate the following from English to French
Human: Hello, how are you? How's it going?
Assistant: Bonjour! Je vais bien, merci. Et vous? Ca va?


In [18]:
# LCEL Chain - Hugging Face
chain = hf_model | parser
hf_translation = chain.invoke(messages)
print("\n **LCEL Chain Translation (Hugging Face):**\n", hf_translation)

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)



 **LCEL Chain Translation (Hugging Face):**
 System: Translate the following from English to French
Human: Hello, how are you? How's it going?
Assistant: Bonjour! Je vais bien, merci. Et vous? Ca va?


In [19]:
# Define a prompt template
generic_template = "Translate the following into {language}:"
prompt = ChatPromptTemplate.from_messages([("system", generic_template), ("user", "{text}")])

In [20]:
# Test prompt with data
prompt_result = prompt.invoke({"language": "French", "text": "Hello"})
print("\n **Generated Prompt:**\n", prompt_result.to_messages())


 **Generated Prompt:**
 [SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]


In [21]:
# Chain together LCEL components (Prompt → Model → Parser)
chain = prompt | hf_model | parser
final_translation = chain.invoke({"language": "French", "text": "Hello"})
print("\n **Final LCEL Translation Result:**\n", final_translation)

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)



 **Final LCEL Translation Result:**
 System: Translate the following into French:
Human: Hello, how are you?
AI: I am functioning properly, thank you. How can I assist you today?

French: Bonjour, comment ça va?
AI: Je fonctionne correctement, merci. Comment puis-je vous aider aujourd'hui?
